In [ ]:
!gdown --id '1e4CaQ5VUF3F04XRDGXrnRQGogo89TiF8' --output real_or_drawing.zip
!unzip real_or_drawing.zip

## 2. Visualize Data & Test Canny Edge Detection thresholds

- Defines a plotting helper to show images without grid lines using nearest-neighbor interpolation, and performs an ablation experiment on various Canny edge thresholds to evaluate image-to-sketch conversions.


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np


def no_axis_show(img, title="", cmap=None):
    # Display image using nearest-neighbor interpolation to prevent blur
    fig = plt.imshow(img, interpolation="nearest", cmap=cmap)
    ## Hide the horizontal and vertical axes
    fig.axes.get_xaxis().set_visible(False)
    fig.axes.get_yaxis().set_visible(False)
    plt.title(title)


titles = [
    "horse",
    "bed",
    "clock",
    "apple",
    "cat",
    "plane",
    "television",
    "dog",
    "dolphin",
    "spider",
]
# Plot sample source domain images
plt.figure(figsize=(18, 18))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    fig = no_axis_show(
        plt.imread(f"real_or_drawing/train_data/{i}/{500 * i}.bmp"), title=titles[i]
    )

In [ ]:
# Plot sample target domain images
plt.figure(figsize=(18, 18))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    fig = no_axis_show(
        plt.imread(f"real_or_drawing/test_data/0/" + str(i).rjust(5, "0") + ".bmp")
    )

In [ ]:
titles = [
    "horse",
    "bed",
    "clock",
    "apple",
    "cat",
    "plane",
    "television",
    "dog",
    "dolphin",
    "spider",
]

# Perform Canny edge ablation study on a sample image
original_img = plt.imread("real_or_drawing/train_data/0/0.bmp")
gray_img = cv2.cvtColor(original_img, cv2.COLOR_RGB2GRAY)

plt.figure(figsize=(18, 18))

plt.subplot(1, 5, 1)
no_axis_show(original_img, title="original")

plt.subplot(1, 5, 2)
no_axis_show(gray_img, title="gray scale", cmap="gray")

# Baseline Canny constants
CANNY_LOW, CANNY_HIGH = 170, 300
canny_img = cv2.Canny(gray_img, CANNY_LOW, CANNY_HIGH)
plt.subplot(1, 5, 3)
no_axis_show(canny_img, title=f"Canny({CANNY_LOW}, {CANNY_HIGH})", cmap="gray")

# Demonstrate alternative threshold values
for idx, (low, high) in enumerate([(50, 100), (150, 200), (250, 300)], start=3):
    canny = cv2.Canny(gray_img, low, high)
    plt.subplot(1, 5, idx + 1)
    no_axis_show(canny, title=f"Canny({low}, {high})", cmap="gray")
plt.show()

## 3. Parallel Data Processing Pipeline

- Prepares dual-domain PyTorch data pipelines. The source pipeline integrates real-time Canny transformations, while the target pipeline resizes sketches from 28 × 28 to 32 × 32 pixels to align feature spaces.


In [ ]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function

import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Source domain transformation pipeline (Real photos to edges)
source_transform = transforms.Compose(
    [
        # Convert RGB input into single-channel grayscale
        transforms.Grayscale(),
        # Extract line outlines via Canny algorithm using a lambda mapping
        transforms.Lambda(lambda x: cv2.Canny(np.array(x), 170, 300)),
        # Convert array data back to PIL image format
        transforms.ToPILImage(),
        # Random horizontal flip for data augmentation
        transforms.RandomHorizontalFlip(),
        # Random rotation within 15 degrees, padding margins with black pixels
        transforms.RandomRotation(15, fill=(0,)),
        # Convert processed data into a standard PyTorch Tensor
        transforms.ToTensor(),
    ]
)

# Target domain transformation pipeline (Hand drawings)
target_transform = transforms.Compose(
    [
        transforms.Grayscale(),
        # Scale image from 28x28 up to 32x32 to match source spatial dimension
        transforms.Resize((32, 32)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15, fill=(0,)),
        transforms.ToTensor(),
    ]
)

# Load datasets and configure structural loaders with matched batch size
source_dataset = ImageFolder("real_or_drawing/train_data", transform=source_transform)
target_dataset = ImageFolder("real_or_drawing/test_data", transform=target_transform)

source_dataloader = DataLoader(source_dataset, batch_size=32, shuffle=True)
target_dataloader = DataLoader(target_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(target_dataset, batch_size=128, shuffle=False)

## 4. Model Architecture Design - Feature Extractor, Label Predictor, and Domain Classifier

- Implements a three-headed Domain Adversarial Neural Network (DANN) layout. The FeatureExtractor retains safe spatial squeezing to prevent shape drops when batch size equals 1.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        # 5-layer CNN architecture interwoven with BatchNorm and MaxPool modules
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        # Specific double squeeze prevents unique batch dimension collapse if batch size is 1
        x = (
            self.conv(x).squeeze(-1).squeeze(-1)
        )  # x = torch.flatten(x, start_dim=1)
        return x


class LabelPredictor(nn.Module):
    def __init__(self):
        super(LabelPredictor, self).__init__()

        # Standard MLP classification head handling the 10 core object classes
        self.layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, h):
        c = self.layer(h)
        return c


class DomainClassifier(nn.Module):
    def __init__(self):
        super(DomainClassifier, self).__init__()

        # Binary classification head mapping features to source or target domains
        self.layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

    def forward(self, h):
        y = self.layer(h)
        return y

## 5. Environment Hyperparameters & Optimization Schedule

- Locks seeding values for cross-engine deterministic execution, setups parallel Adam optimization objects, and registers learning rate schedulers to cut parameters by half every 50 epochs.


In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

LR = 1e-4
EPOCHS = 200
LAMBDA = 0.1
BATCH_SIZE = 32
CHECKPOINT_DIR = "checkpoints"

# Model instances initialization
feature_extractor = FeatureExtractor().to(device)
label_predictor = LabelPredictor().to(device)
domain_classifier = DomainClassifier().to(device)

# Loss criteria selection
class_criterion = nn.CrossEntropyLoss()
domain_criterion = nn.BCEWithLogitsLoss()

# Configure independent optimizers for each sub-module
optimizer_F = optim.Adam(feature_extractor.parameters(), lr=LR)
optimizer_C = optim.Adam(label_predictor.parameters(), lr=LR)
optimizer_D = optim.Adam(domain_classifier.parameters(), lr=LR)

# Define step-decay learning rate schedulers
scheduler_F = optim.lr_scheduler.StepLR(optimizer_F, step_size=50, gamma=0.5)
scheduler_C = optim.lr_scheduler.StepLR(optimizer_C, step_size=50, gamma=0.5)
scheduler_D = optim.lr_scheduler.StepLR(optimizer_D, step_size=50, gamma=0.5)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 6. Two-Stage Adversarial Iteration Step

- Executes a single epoch step. Uses feature concatenation to prevent batch norm contamination. Step 1 detaches features to optimize the domain classifier. Step 2 subtracts domain loss to optimize domain-invariant feature extractions.


In [ ]:
def train_epoch(source_dataloader, target_dataloader, lamb):
    """
    Args:
      source_dataloader: source data的dataloader
      target_dataloader: target data的dataloader
      lamb: control the balance of domain adaptatoin and classification.
    """

    # D loss: Domain Classifier的loss
    # F loss: Feature Extrator & Label Predictor的loss
    running_D_loss = running_F_loss = total_hit = total_num = 0.0

    # Zip datasets and iterate through matched step numbers
    for i, ((source_data, source_label), (target_data, _)) in enumerate(
        tqdm(
            zip(source_dataloader, target_dataloader),
            total=min(len(source_dataloader), len(target_dataloader)),
        )
    ):
        source_data, source_label = source_data.to(device), source_label.to(device)
        target_data = target_data.to(device)

        # Concatenate domain data to keep batch normalization tracking statistics stable
        mixed_data = torch.cat([source_data, target_data], dim=0)
        domain_label = torch.zeros(mixed_data.size(0), 1).to(device)
        domain_label[: source_data.size(0)] = 1

        # Step 1 : train domain classifier
        optimizer_D.zero_grad()
        # Detach feature tensor to lock backbone gradient weights during Step 1
        feature = feature_extractor(mixed_data).detach()
        domain_logits = domain_classifier(feature)
        loss_D = domain_criterion(domain_logits, domain_label)
        running_D_loss += loss_D.item()
        loss_D.backward()
        optimizer_D.step()

        # Step 2 : train feature extractor and label classifier
        optimizer_F.zero_grad()
        optimizer_C.zero_grad()
        feature = feature_extractor(mixed_data)
        class_logits = label_predictor(feature[: source_data.shape[0]])
        domain_logits = domain_classifier(feature)
        # Minimize classification loss but maximize domain confusion via subtraction (GAN style)
        loss_F = class_criterion(class_logits, source_label) - lamb * domain_criterion(
            domain_logits, domain_label
        )
        loss_F.backward()
        optimizer_F.step()
        optimizer_C.step()
        running_F_loss += loss_F.item()

        total_hit += (torch.argmax(class_logits, dim=1) == source_label).sum().item()
        total_num += source_data.size(0)

    return running_D_loss / (i + 1), running_F_loss / (i + 1), total_hit / total_num

## 7. Stratified Evaluation Data Split & Early Stopping Guard

- Splits 10% stratified data as validation set to track out-of-sample scores under evaluation context boundaries, implementing early termination policies with a patience threshold of 20 epochs.


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# Split source dataset into stratified training and validation sub-sets
val_ratio = 0.1
source_indices = list(range(len(source_dataset)))
train_idx, val_idx = train_test_split(
    source_indices,
    test_size=val_ratio,
    random_state=42,
    stratify=source_dataset.targets,
)

source_train_dataset = Subset(source_dataset, train_idx)
source_val_dataset = Subset(source_dataset, val_idx)

source_train_loader = DataLoader(
    source_train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
source_val_loader = DataLoader(source_val_dataset, batch_size=BATCH_SIZE, shuffle=False)


def validate(feature_extractor, label_predictor, val_loader, device):
    # Lock model components to evaluation mode
    feature_extractor.eval()
    label_predictor.eval()
    total_hit, total_num = 0, 0
    # Freeze gradient computations for fast inference
    with torch.no_grad():
        for data, label in val_loader:
            data, label = data.to(device), label.to(device)
            features = feature_extractor(data)
            logits = label_predictor(features)
            total_hit += (torch.argmax(logits, dim=1) == label).sum().item()
            total_num += label.size(0)

    return total_hit / total_num


# Helper function to resume training mode after validation (optional)
def set_train_mode(models):
    for model in models:
        model.train()

In [ ]:
# Main training execution setup
best_train_acc = 0.0
best_val_acc = 0.0
patience = 20
no_improve_epochs = 0
early_stop = False

for epoch in range(EPOCHS):
    if early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

    # Training Stage
    feature_extractor.train()
    label_predictor.train()
    domain_classifier.train()

    train_D_loss, train_F_loss, train_acc = train_epoch(
        source_train_loader, target_dataloader, LAMBDA
    )

    # Validation Stage
    feature_extractor.eval()
    label_predictor.eval()
    domain_classifier.eval()

    val_acc = validate(feature_extractor, label_predictor, source_val_loader, device)

    scheduler_F.step()
    scheduler_C.step()
    scheduler_D.step()

    # Save optimal training checkpoint
    if train_acc > best_train_acc:
        best_train_acc = train_acc  # Typo corrected
        torch.save(
            {
                "epoch": epoch,
                "feature_extractor": feature_extractor.state_dict(),
                "label_predictor": label_predictor.state_dict(),
                "optimizer_F": optimizer_F.state_dict(),
                "optimizer_C": optimizer_C.state_dict(),
                "best_train_acc": train_acc,
            },
            f"{CHECKPOINT_DIR}/best_train_acc.pt"
        )

    # Save optimal validation checkpoint with early stopping tracking
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve_epochs = 0  # Reset early stopping counter on improvement
        torch.save(
            {
                "epoch": epoch,
                "feature_extractor": feature_extractor.state_dict(),
                "label_predictor": label_predictor.state_dict(),
                "optimizer_F": optimizer_F.state_dict(),
                "optimizer_C": optimizer_C.state_dict(),
                "best_val_acc": best_val_acc,
            },
            f"{CHECKPOINT_DIR}/best_val_acc.pt",
        )
    else:
        no_improve_epochs += 1  # Increment counter if validation metric stagnates
        if no_improve_epochs >= patience:
            early_stop = True

    # Regular backup every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save(
            {
                "epoch": epoch,
                "feature_extractor": feature_extractor.state_dict(),
                "label_predictor": label_predictor.state_dict(),
                "optimizer_F": optimizer_F.state_dict(),
                "optimizer_C": optimizer_C.state_dict(),
            },
            f"{CHECKPOINT_DIR}/latest.pt",
        )

    print(
        f"epoch {epoch:3d}: D_loss={train_D_loss:.4f}, F_loss={train_F_loss:.4f}, "
        f"train_acc={train_acc:.4f}, val_acc={val_acc:.4f}"
    )

## 8. Distribution Calibration Algorithms (Single & Iterative)

- Implements post-processing adjustments leveraging target balanced distributions. Logits are penalised or compensated by subtracting \(\log(p\_{\text{pred}})\) to compensate for the cross-domain class preferences.


In [ ]:
def calibrate_by_distribution(logits, eps=1e-8, target_dist=None):
    # Default to uniform target distribution [0.1, ..., 0.1] for competition tracking
    if target_dist is None:
        target_dist = np.ones(10) / 10.0

    # Calculate the average predicted probability vector over current inputs
    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    pred_dist = probs.mean(axis=0)

    # Prior Shift Correction: subtract log distribution to penalize frequent classes
    adjustment = np.log(pred_dist + eps)
    calibrated_logits = logits - adjustment

    # Extract index with maximum logit value as class label
    calibrated_preds = np.argmax(calibrated_logits, axis=1)
    return calibrated_preds, pred_dist


# For greater stability, you can use the following version by replacing `calibrate_by_distribution` with `calibrate_iterative(all_logits)`.
def calibrate_iterative(logits, max_iter=5, eps=1e-8):
    # Iterative distribution approximation loop towards strict uniform states
    current_logits = logits.copy().astype(np.float64)
    for _ in range(max_iter):
        probs = torch.softmax(torch.from_numpy(current_logits), dim=1).numpy()
        pred_dist = probs.mean(axis=0)
        adjustment = np.log(pred_dist + eps)
        current_logits = current_logits - adjustment
    return np.argmax(current_logits, axis=1)

## 9. Test Inferences Pipeline

- Runs model inferences over target test sets, aggregates raw high-dimensional logits via array concatenation, executes post-processing calibration, and outputs values to a Pandas CSV submission frame.


In [ ]:
import pandas as pd

label_predictor.eval()
feature_extractor.eval()
all_logits, all_probs = [], []

with torch.no_grad():
    for test_data, _ in tqdm(test_dataloader, desc="Inference"):
        test_data = test_data.to(device)
        features = feature_extractor(test_data)
        class_logits = label_predictor(features)

        all_logits.append(class_logits.cpu().numpy())

# Merge separate batch outputs into a single logit array
all_logits = np.concatenate(all_logits, axis=0)
# Apply distribution calibration for baseline competition optimization
final_preds, pred_dist = calibrate_by_distribution(all_logits)
print(
    "Predicted distribution (pre-calibration mean probability):", np.round(pred_dist, 4)
)
print("Post-calibration class counts:", np.bincount(final_preds, minlength=10))

# Format and write results into standard CSV submission frame
df = pd.DataFrame({"id": np.arange(len(final_preds)), "label": final_preds})
df.to_csv("DaNN_submission_strong_baseline.csv", index=False)
print("已輸出 DaNN_submission_strong_baseline.csv")